# Calendar Dimension Table

Builds the standard calendar dimension `azuredb_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
%sql
CREATE OR REPLACE TABLE azuredb_caresync.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)              AS date_key,
  d                                                    AS date,
  -- Day
  day(d)                                               AS day_of_month,
  extract(DAYOFWEEK_ISO FROM d)                        AS day_of_week,        -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                               AS day_name,
  date_format(d, 'EEE')                                AS day_name_short,
  dayofyear(d)                                         AS day_of_year,
  CASE WHEN extract(DAYOFWEEK_ISO FROM d) IN (6, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  CASE WHEN extract(DAYOFWEEK_ISO FROM d) IN (6, 7) THEN FALSE ELSE TRUE END AS is_weekday,
  -- Week (ISO 8601, weeks start on Monday)
  weekofyear(d)                                        AS week_of_year,
  extract(YEAROFWEEK FROM d)                           AS iso_year,
  CAST(date_trunc('WEEK', d) AS DATE)                  AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)     AS week_end_date,
  CAST(ceil(day(d) / 7.0) AS INT)                      AS week_of_month,
  -- Month
  month(d)                                             AS month,
  date_format(d, 'MMMM')                               AS month_name,
  date_format(d, 'MMM')                                AS month_name_short,
  date_format(d, 'yyyy-MM')                            AS year_month,
  year(d) * 100 + month(d)                             AS year_month_key,
  CAST(date_trunc('MONTH', d) AS DATE)                 AS month_start_date,
  last_day(d)                                          AS month_end_date,
  day(last_day(d))                                     AS days_in_month,
  -- Quarter
  quarter(d)                                           AS quarter,
  concat('Q', quarter(d))                              AS quarter_name,
  concat(year(d), '-Q', quarter(d))                    AS year_quarter,
  CAST(date_trunc('QUARTER', d) AS DATE)               AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS quarter_end_date,
  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END            AS half_year,
  concat(year(d), '-H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END) AS year_half,
  -- Year
  year(d)                                              AS year,
  CAST(date_trunc('YEAR', d) AS DATE)                  AS year_start_date,
  make_date(year(d), 12, 31)                           AS year_end_date,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 THEN TRUE ELSE FALSE END AS is_leap_year,
  CASE WHEN d = last_day(d) THEN TRUE ELSE FALSE END   AS is_month_end,
  CASE WHEN d = date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) THEN TRUE ELSE FALSE END AS is_quarter_end,
  CASE WHEN month(d) = 12 AND day(d) = 31 THEN TRUE ELSE FALSE END AS is_year_end
FROM dates
ORDER BY d;

In [0]:
%sql
SELECT
  count(*)                  AS total_rows,
  count(DISTINCT date_key)  AS distinct_date_keys,
  min(date)                 AS min_date,
  max(date)                 AS max_date
FROM azuredb_caresync.silver.calendar;

In [0]:
%sql
SELECT * FROM azuredb_caresync.silver.calendar
WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-05'
ORDER BY date;